# Track 2 — Transformer Diffusion Model
### Empirical memorization & generation quality on real data (DiT + DDPM)

**Purpose:** Test whether Track 1's theoretical insights hold for practical architectures on real data.

| Quantity | Symbol | Definition |
|---|---|---|
| Generation onset | τ_gen | Training step when FID drops below threshold |
| Memorization onset | τ_mem | Training step when memo ratio exceeds threshold |
| Optimal latent dim | d* | Value of d_latent minimising FID_test |
| Intrinsic dim | d_int | FLIPD estimate of dataset manifold dimension |

**Hypothesis:** `d* ≈ d_int`; larger `d_latent` leads to earlier `τ_mem`.  
Runs on **Kaggle** (T4/P100) or **Google Cloud** (Vertex AI / Colab Enterprise).  
Recommended: at least one GPU with 16 GB VRAM for DiT-B on CIFAR-10.

---
## 0 · Environment setup

In [ ]:
import os, sys, subprocess

os.environ.update({
    'TF_CPP_MIN_LOG_LEVEL':         '3',
    'TF_ENABLE_ONEDNN_OPTS':         '0',
    'CUDA_MODULE_LOADING':           'LAZY',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'GRPC_VERBOSITY':                'ERROR',
    'GLOG_minloglevel':              '3',
})
import warnings, logging
warnings.filterwarnings('ignore', category=UserWarning)
for _log in ('tensorflow', 'jax', 'absl'): logging.getLogger(_log).setLevel(logging.ERROR)

IN_KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB  = 'google.colab' in sys.modules
IN_VERTEX = os.environ.get('VERTEX_AI_USER_MANAGED_NOTEBOOKS') == 'true'
IN_GCP    = IN_COLAB or IN_VERTEX or os.environ.get('CLOUD_ML_PROJECT_ID') is not None
ENV       = 'kaggle' if IN_KAGGLE else ('gcp' if IN_GCP else 'local')
print(f'Environment: {ENV.upper()}')

if IN_KAGGLE:
    OUT_DIR = '/kaggle/working/track2'
elif IN_VERTEX:
    GCS_BUCKET = os.environ.get('AIP_MODEL_DIR', '')
    OUT_DIR = GCS_BUCKET if GCS_BUCKET else '/home/jupyter/track2'
else:
    OUT_DIR = './track2'

CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints')
os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)
print(f'Output → {OUT_DIR}')

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for pkg, imp in [('scipy','scipy'), ('einops','einops')]:
    try:
        __import__(imp)
    except ImportError:
        pip(pkg)

print('Deps ready.')

In [ ]:
import torch, numpy as np, random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    # Enable TF32 on Ampere GPUs for ~2x throughput
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

---
## 1 · Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MULTI-RUN SETUP: Change RUN_ID for each Kaggle session.
#    Run 1 → trains VAEs + DiT on n=1000   (~4 h on T4)
#    Run 2 → loads VAEs + DiT on n=5000    (~4 h on T4)
#    Run 3 → loads VAEs + DiT on n=50000   (~4 h on T4)
#  After all 3 runs, the merge cell combines results and plots the dashboard.
# ══════════════════════════════════════════════════════════════════════════════

RUN_ID = 1  # ← change to 2 or 3 for subsequent Kaggle sessions

RUN_SCHEDULE = {
    1: [1000],
    2: [5000, 10000],
    3: [25000],
    4: [50000],
}

# For Runs 2+: set this to the path where you uploaded Run 1's checkpoints.
# On Kaggle, uploaded datasets appear under /kaggle/input/<dataset-name>/...
# Set to None on Run 1 (checkpoints don't exist yet).
INPUT_CKPT_DIR = None  # e.g. '/kaggle/input/datasets/kynnet/checkpoints'

CFG = dict(
    # ── Data ──────────────────────────────────────────────────────────────────
    dataset        = 'cifar10',  # 'cifar10' | 'celeba' | 'ffhq'
    image_size     = 32,         # spatial resolution (32 for CIFAR, 64/256 for CelebA/FFHQ)
    n_classes      = 10,         # set 0 for unconditional

    # ── Latent space sweep ────────────────────────────────────────────────────
    # Each entry is the number of latent channels c;
    # latent shape = (c, image_size//8, image_size//8) following standard VAE downscale
    latent_channels = [1, 2, 4, 8, 16],
    latent_spatial  = 4,    # h=w=4 for 32×32 images with 8× downscale
                            # → d_latent = c × 4 × 4

    # ── DiT architecture ──────────────────────────────────────────────────────
    dit_variant    = 'S',   # 'S' | 'B' | 'L' | 'XL'
    patch_size     = 2,     # p×p patchification of the latent spatial grid

    # ── Flow matching ─────────────────────────────────────────────────────────
    objective      = 'flow_matching',
    fm_sample_steps = 50,        # Euler ODE steps for sampling

    # ── Training ──────────────────────────────────────────────────────────────
    total_steps    = 80_000,     # gradient steps per (d_latent, n) run
    batch_size     = 128,
    lr             = 1e-4,
    ema_decay      = 0.9999,
    grad_clip      = 1.0,
    mixed_prec     = True,       # AMP fp16/bf16

    # ── Dataset size sweep ────────────────────────────────────────────────────
    n_train_grid   = RUN_SCHEDULE[RUN_ID],

    # ── VAE ───────────────────────────────────────────────────────────────────
    vae_epochs     = 20,         # epochs for ConvVAE pre-training

    # ── Evaluation ────────────────────────────────────────────────────────────
    loss_every     = 500,        # record training loss every N steps
    eval_every     = 5_000,      # measure metrics every N steps
    n_fid_samples  = 5_000,      # samples for FID at each checkpoint
    fid_threshold  = 50.0,       # τ_gen: FID below this → generation has 'started'
    memo_threshold = 0.075,       # τ_mem: NN memo rate above this → memorisation
    memo_nn_thresh = 0.5,        # d1/d2 ratio threshold per sample (Carlini/Gu)

    # ── FLIPD ─────────────────────────────────────────────────────────────────
    flipd_n_probes = 15,          # Hutchinson probes for velocity Jacobian trace
    flipd_t_vals   = [0.005, 0.01, 0.05, 0.1],  # noise levels t ∈ [0,1] for FLIPD evaluation
    flipd_primary_t = 0.01,      # t value used for d_intrinsic estimate

    # ── Misc ──────────────────────────────────────────────────────────────────
    seed           = 42,
    num_workers    = 4 if DEVICE == 'cuda' else 0,
    save_ckpt      = True,
    out_dir        = OUT_DIR,
    ckpt_dir       = CKPT_DIR,
)

# DiT variant specs ────────────────────────────────────────────────────────────
DIT_SPECS = {
    'S':  dict(d_model=384,  depth=12, n_heads=6),
    'B':  dict(d_model=768,  depth=12, n_heads=12),
    'L':  dict(d_model=1024, depth=24, n_heads=16),
    'XL': dict(d_model=1152, depth=28, n_heads=16),
}
CFG['dit_spec'] = DIT_SPECS[CFG['dit_variant']]

# Reproducibility
random.seed(CFG['seed']); np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if DEVICE == 'cuda': torch.cuda.manual_seed_all(CFG['seed'])

print(f'DiT-{CFG["dit_variant"]} | {CFG["objective"]} | {CFG["fm_sample_steps"]} Euler steps')
print(f'Latent channel sweep: {CFG["latent_channels"]}')
print(f'd_latent per run: {[c * CFG["latent_spatial"]**2 for c in CFG["latent_channels"]]}')

---
## 2 · Data loading

In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

def get_dataloaders(dataset_name: str, image_size: int, batch_size: int):
    # /kaggle/input is read-only — always download/cache to a writable dir
    write_root = '/kaggle/working/datasets' if IN_KAGGLE else './data'
    os.makedirs(write_root, exist_ok=True)

    tfm = T.Compose([
        T.Resize(image_size),
        T.CenterCrop(image_size),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize([0.5]*3, [0.5]*3),   # → [-1, 1]
    ])

    if dataset_name == 'cifar10':
        # Prefer the pre-attached Kaggle dataset input (no download needed);
        # fall back to downloading into the writable working dir.
        kaggle_cifar = '/kaggle/input/cifar-10-python'
        if IN_KAGGLE and os.path.isdir(kaggle_cifar):
            # torchvision expects the .tar.gz to already be extracted here
            root = kaggle_cifar
            download = False
        else:
            root = write_root
            download = True
        train_ds = torchvision.datasets.CIFAR10(root, train=True,  download=download, transform=tfm)
        test_ds  = torchvision.datasets.CIFAR10(root, train=False, download=download, transform=tfm)

    elif dataset_name == 'celeba':
        # On Kaggle: add the 'CelebA Faces Attributes Dataset' input
        celeba_root = '/kaggle/input/celeba-dataset' if IN_KAGGLE else './data/celeba'
        train_ds = torchvision.datasets.CelebA(celeba_root, split='train', download=False, transform=tfm)
        test_ds  = torchvision.datasets.CelebA(celeba_root, split='test',  download=False, transform=tfm)

    elif dataset_name == 'ffhq':
        # On Kaggle: add the FFHQ dataset input
        ffhq_root = '/kaggle/input/ffhq-dataset/images' if IN_KAGGLE else './data/ffhq'
        full_ds   = torchvision.datasets.ImageFolder(ffhq_root, transform=tfm)
        n_train   = int(0.9 * len(full_ds))
        indices   = list(range(len(full_ds)))
        train_ds  = Subset(full_ds, indices[:n_train])
        test_ds   = Subset(full_ds, indices[n_train:])

    else:
        raise ValueError(f'Unknown dataset: {dataset_name}')

    kw = dict(batch_size=batch_size, num_workers=CFG['num_workers'],
              pin_memory=(DEVICE=='cuda'), drop_last=True)
    train_loader = DataLoader(train_ds, shuffle=True,  **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
    return train_loader, test_loader, train_ds, test_ds


train_loader, test_loader, train_ds, test_ds = get_dataloaders(
    CFG['dataset'], CFG['image_size'], CFG['batch_size']
)
print(f'Train: {len(train_ds):,}  |  Test: {len(test_ds):,}')
print(f'Batches per epoch: {len(train_loader):,}')

---
## 3 · Diffusion schedule

In [ ]:
import math


class FlowSchedule:
    """
    Conditional flow matching with linear interpolation path.
    x_t = (1-t)*x_0 + t*noise,  velocity = noise - x_0
    t=0 → clean data,  t=1 → pure noise.
    """
    def interpolate(self, x0: torch.Tensor, t: torch.Tensor, noise=None):
        """Interpolate between data and noise; return (x_t, velocity)."""
        if noise is None:
            noise = torch.randn_like(x0)
        t_ = t.view(-1, *([1] * (x0.dim() - 1)))
        x_t = (1 - t_) * x0 + t_ * noise
        velocity = noise - x0
        return x_t, velocity


schedule = FlowSchedule()
print(f'Flow matching schedule ready  (linear interpolation, t ∈ [0, 1])')

---
## 4 · DiT model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

# ── Timestep sinusoidal embedding ─────────────────────────────────────────────
class SinusoidalEmbed(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freq = torch.exp(-math.log(10000) *
                         torch.arange(half, device=t.device) / (half - 1))
        emb  = (t.float() * 1000).unsqueeze(1) * freq.unsqueeze(0)
        return torch.cat([emb.sin(), emb.cos()], dim=-1)   # (B, dim)


# ── AdaLN-Zero modulation ─────────────────────────────────────────────────────
class AdaLNZero(nn.Module):
    """
    Conditioning via Adaptive LayerNorm with zero-init final projection.
    Produces scale (γ) and shift (β) from conditioning vector c.
    """
    def __init__(self, d_model: int, d_cond: int, n_params: int = 6):
        super().__init__()
        self.norm   = nn.LayerNorm(d_model, elementwise_affine=False, eps=1e-6)
        self.proj   = nn.Linear(d_cond, n_params * d_model)
        # Zero-init so network starts as identity
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)
        self.n_params = n_params

    def forward(self, x: torch.Tensor, c: torch.Tensor):
        """Returns normalised x and a list of n_params modulation tensors."""
        params = self.proj(F.silu(c))   # (B, n_params * d_model)
        params = params.chunk(self.n_params, dim=-1)   # n_params × (B, d_model)
        return self.norm(x), params


# ── Multi-Head Self-Attention ──────────────────────────────────────────────────
class MHSA(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model)
        self.scale = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(2)          # each: (B, N, H, D)
        q = q.transpose(1, 2)            # (B, H, N, D)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        # Use scaled_dot_product_attention if available (torch ≥ 2.0)
        if hasattr(F, 'scaled_dot_product_attention'):
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        else:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            attn = attn.softmax(dim=-1)
            out  = attn @ v
        out = out.transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


# ── DiT Block ─────────────────────────────────────────────────────────────────
class DiTBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_cond: int, mlp_ratio: float = 4.0):
        super().__init__()
        self.adaLN = AdaLNZero(d_model, d_cond, n_params=6)
        self.attn  = MHSA(d_model, n_heads)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, d_model),
        )

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        x_norm, (γ1, β1, α1, γ2, β2, α2) = self.adaLN(x, c)
        x = x + α1.unsqueeze(1) * self.attn(
            (1 + γ1.unsqueeze(1)) * x_norm + β1.unsqueeze(1))
        x_norm2 = nn.functional.layer_norm(x, x.shape[-1:])
        x = x + α2.unsqueeze(1) * self.mlp(
            (1 + γ2.unsqueeze(1)) * x_norm2 + β2.unsqueeze(1))
        return x


# ── Full DiT ──────────────────────────────────────────────────────────────────
class DiT(nn.Module):
    """
    Diffusion Transformer following Peebles & Xie 2023.

    Args:
        in_channels : latent channels c
        spatial_size: latent h = w (e.g. 4 for 32×32 images with 8× VAE)
        patch_size  : p (patchify the latent spatial grid)
        d_model, depth, n_heads: DiT variant dimensions
        n_classes   : number of conditioning classes (0 = unconditional)
    """
    def __init__(
        self,
        in_channels : int,
        spatial_size: int,
        patch_size  : int,
        d_model     : int,
        depth       : int,
        n_heads     : int,
        n_classes   : int = 0,
        d_cond      : int = None,
    ):
        super().__init__()
        self.in_channels  = in_channels
        self.spatial_size = spatial_size
        self.patch_size   = patch_size
        self.n_patches    = (spatial_size // patch_size) ** 2
        self.patch_dim    = in_channels * patch_size * patch_size
        d_cond            = d_cond or d_model

        # ── Input ────────────────────────────────────────────────────────────
        self.patch_embed = nn.Linear(self.patch_dim, d_model)
        self.pos_embed   = nn.Parameter(
            torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # ── Conditioning ─────────────────────────────────────────────────────
        self.t_embed = nn.Sequential(
            SinusoidalEmbed(d_model),
            nn.Linear(d_model, d_cond),
            nn.SiLU(),
            nn.Linear(d_cond, d_cond),
        )
        if n_classes > 0:
            self.class_embed = nn.Embedding(n_classes + 1, d_cond)  # +1 for null
        else:
            self.class_embed = None

        # ── Transformer blocks ────────────────────────────────────────────────
        self.blocks = nn.ModuleList([
            DiTBlock(d_model, n_heads, d_cond) for _ in range(depth)
        ])

        # ── Output ────────────────────────────────────────────────────────────
        self.final_norm = nn.LayerNorm(d_model, elementwise_affine=False, eps=1e-6)
        self.final_adaLN_proj = nn.Linear(d_cond, 2 * d_model)
        nn.init.zeros_(self.final_adaLN_proj.weight)
        nn.init.zeros_(self.final_adaLN_proj.bias)
        self.final_proj = nn.Linear(d_model, self.patch_dim)
        nn.init.zeros_(self.final_proj.weight)
        nn.init.zeros_(self.final_proj.bias)

    def patchify(self, z: torch.Tensor) -> torch.Tensor:
        """(B,C,H,W) → (B, n_patches, patch_dim)"""
        p = self.patch_size
        return rearrange(z, 'b c (h p1) (w p2) -> b (h w) (c p1 p2)', p1=p, p2=p)

    def unpatchify(self, x: torch.Tensor) -> torch.Tensor:
        """(B, n_patches, patch_dim) → (B,C,H,W)"""
        p  = self.patch_size
        hw = int(self.n_patches ** 0.5)
        return rearrange(x, 'b (h w) (c p1 p2) -> b c (h p1) (w p2)',
                         h=hw, w=hw, p1=p, p2=p)

    def forward(
        self,
        z_t    : torch.Tensor,          # (B, C, H, W) noisy latent
        t      : torch.Tensor,          # (B,) integer timesteps
        labels : torch.Tensor = None,   # (B,) class labels or None
    ) -> torch.Tensor:
        # Patchify + embed
        x = self.patchify(z_t)          # (B, N, patch_dim)
        x = self.patch_embed(x)         # (B, N, d_model)
        x = x + self.pos_embed

        # Conditioning vector
        c = self.t_embed(t)             # (B, d_cond)
        if self.class_embed is not None and labels is not None:
            c = c + self.class_embed(labels)

        # Transformer blocks
        for block in self.blocks:
            x = block(x, c)

        # Final AdaLN + project back to patch_dim
        shift, scale = self.final_adaLN_proj(F.silu(c)).chunk(2, dim=-1)
        x = self.final_norm(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        x = self.final_proj(x)          # (B, N, patch_dim)
        return self.unpatchify(x)       # (B, C, H, W)


def build_dit(in_channels: int) -> DiT:
    spec = CFG['dit_spec']
    model = DiT(
        in_channels  = in_channels,
        spatial_size = CFG['latent_spatial'],
        patch_size   = CFG['patch_size'],
        d_model      = spec['d_model'],
        depth        = spec['depth'],
        n_heads      = spec['n_heads'],
        n_classes    = CFG['n_classes'],
    ).to(DEVICE)
    return model


# Quick sanity check
_c  = CFG['latent_channels'][0]
_sp = CFG['latent_spatial']
_m  = build_dit(_c)
_z  = torch.randn(2, _c, _sp, _sp, device=DEVICE)
_t  = torch.rand(2, device=DEVICE)
_out = _m(_z, _t)
print(f'DiT-{CFG["dit_variant"]} smoke test OK  '
      f'in={tuple(_z.shape)} → out={tuple(_out.shape)}')
n_params = sum(p.numel() for p in _m.parameters()) / 1e6
print(f'Parameters: {n_params:.1f} M')
del _m, _z, _t, _out

---
## 5 · EMA, DDIM sampler & FID utilities

In [ ]:
import copy
from scipy.linalg import sqrtm

# ── Exponential Moving Average ─────────────────────────────────────────────────
class EMA:
    """Maintains a shadow copy of model weights via EMA."""
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay  = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(self.decay).add_(m.data, alpha=1 - self.decay)

    def __call__(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)


# ── Euler ODE sampler for flow matching ───────────────────────────────────────
@torch.no_grad()
def euler_sample(
    model    : nn.Module,
    shape    : tuple,
    n_steps  : int = 50,
    labels   : torch.Tensor = None,
    cfg_scale: float = 1.0,
    x_init   : torch.Tensor = None,
) -> torch.Tensor:
    """Euler ODE sampling for flow matching (t=1 noise -> t=0 data)."""
    model.eval()
    device = next(model.parameters()).device
    dt = 1.0 / n_steps
    x = x_init.clone() if x_init is not None else torch.randn(shape, device=device)

    for i in range(n_steps):
        t_val = 1.0 - i * dt
        t_batch = torch.full((shape[0],), t_val, device=device)

        v = model(x, t_batch, labels)

        if cfg_scale > 1.0 and labels is not None:
            null_labels = torch.full_like(labels, CFG['n_classes'])
            v_null = model(x, t_batch, null_labels)
            v = v_null + cfg_scale * (v - v_null)

        x = x - v * dt

    return x


# ── FID (feature-space Fréchet distance) ─────────────────────────────────────
def compute_fid(real: np.ndarray, fake: np.ndarray, eps: float = 1e-6) -> float:
    mu_r, mu_f = real.mean(0), fake.mean(0)
    C_r = np.cov(real, rowvar=False)
    C_f = np.cov(fake, rowvar=False)
    # Regularise to avoid singular/near-singular matrices (happens at early
    # training when generated samples are nearly identical)
    C_r += np.eye(C_r.shape[0]) * eps
    C_f += np.eye(C_f.shape[0]) * eps
    diff = mu_r - mu_f
    covmean = sqrtm(C_r @ C_f).real
    # Clip any residual imaginary noise
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(C_r + C_f - 2 * covmean))


def extract_features(loader_or_array, n_max=5000) -> np.ndarray:
    """
    Flatten spatial dims of a dataset into feature vectors for FID.
    Works on DataLoader or numpy array.
    """
    feats = []
    if isinstance(loader_or_array, np.ndarray):
        return loader_or_array.reshape(len(loader_or_array), -1)[:n_max]
    for batch in loader_or_array:
        imgs = batch[0] if isinstance(batch, (list,tuple)) else batch
        feats.append(imgs.reshape(imgs.size(0), -1).numpy())
        if sum(len(f) for f in feats) >= n_max:
            break
    return np.concatenate(feats, axis=0)[:n_max]


# ── FLIPD LID via Hutchinson trace estimator (Kamkari et al. 2024) ────────────
def flipd_lid_dit(
    model   : nn.Module,
    z_0     : torch.Tensor,
    t_val   : float = 0.1,
    n_probes: int = 5,
    labels  : torch.Tensor = None,
) -> torch.Tensor:
    """LID(x,t) ≈ d − t · tr(∂ε̂/∂x_t) estimated via Hutchinson (flow matching)."""
    device = next(model.parameters()).device
    d = z_0[0].numel()

    noise = torch.randn_like(z_0)
    z_t = (1 - t_val) * z_0 + t_val * noise
    t_vec = torch.full((z_0.shape[0],), t_val, device=device)

    traces = []
    for _ in range(n_probes):
        z_in = z_t.detach().requires_grad_(True)
        v_pred = model(z_in, t_vec, labels)
        eps_pred = (1 - t_val) * v_pred + z_in
        probe = torch.randint(0, 2, z_in.shape, device=device).float() * 2 - 1
        (grad_eps,) = torch.autograd.grad(
            outputs=eps_pred, inputs=z_in,
            grad_outputs=probe, create_graph=False,
        )
        traces.append((probe * grad_eps).flatten(1).sum(dim=1))

    avg_trace = torch.stack(traces).mean(dim=0)
    lid = d - t_val * avg_trace
    return lid.detach()


# ── Nearest-neighbour memorisation ratio (Carlini / Gu criterion) ─────────────
def nn_memorization_ratio(
    generated: np.ndarray,
    training : np.ndarray,
    threshold: float = 0.5,
) -> tuple:
    """Per-sample d1/d2 ratio and overall memorisation rate."""
    from sklearn.neighbors import NearestNeighbors
    nn_model = NearestNeighbors(n_neighbors=2, metric='euclidean').fit(training)
    dists, _ = nn_model.kneighbors(generated)
    d1, d2 = dists[:, 0], dists[:, 1]
    ratios = d1 / np.maximum(d2, 1e-10)
    return ratios, float((ratios < threshold).mean())


# ── Eigenvalue / spectral-gap analysis of latent covariance ───────────────────
def eigenvalue_analysis(data: np.ndarray) -> dict:
    """Eigenvalues of sample covariance + spectral gap statistics."""
    cov = np.cov(data, rowvar=False)
    eigs = np.sort(np.linalg.eigvalsh(cov))[::-1].astype(np.float64)
    if len(eigs) > 1:
        gaps = eigs[:-1] - eigs[1:]
        gap_idx = int(np.argmax(gaps))
        gap_val = float(gaps[gap_idx])
    else:
        gap_idx, gap_val = 0, 0.0
    effective_rank = float(np.sum(eigs > eigs[0] * 0.01))
    return {'eigenvalues': eigs, 'spectral_gap': gap_val,
            'gap_position': gap_idx, 'effective_rank': effective_rank}


# ── Inception-v3 feature extractor for true FID ──────────────────────────────
_inception_model = None

def _get_inception():
    global _inception_model
    if _inception_model is None:
        from torchvision.models import inception_v3, Inception_V3_Weights
        m = inception_v3(weights=Inception_V3_Weights.DEFAULT).to(DEVICE)
        m.fc = nn.Identity()
        m.eval()
        for p in m.parameters():
            p.requires_grad_(False)
        _inception_model = m
        print('  Inception-v3 loaded (2048-dim features).', flush=True)
    return _inception_model


@torch.no_grad()
def inception_features(images: torch.Tensor, batch_size: int = 64) -> np.ndarray:
    """Extract 2048-dim Inception-v3 pool features from images in [-1, 1]."""
    model = _get_inception()
    mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
    feats = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size].to(DEVICE)
        batch = (batch + 1) / 2                          # [-1,1] → [0,1]
        batch = F.interpolate(batch, size=(299, 299),
                              mode='bilinear', align_corners=False)
        batch = (batch - mean) / std                     # ImageNet normalise
        out = model(batch)
        feats.append(out.cpu().numpy())
    return np.concatenate(feats)


print('EMA, DDIM, FID, FLIPD, NN-memo, Inception, eigenvalue utilities defined.')

---
## 6 · Synthetic VAE encoder/decoder

> For the latent sweep we need a VAE that maps images ↔ latent tensors of shape `(c, spatial, spatial)`.
> We use a lightweight convolutional VAE. For real experiments you can swap this for a
> pretrained SD-VAE (stabilityai/sd-vae-ft-ema) by uncommenting the HuggingFace block below.

In [ ]:
class ConvVAE(nn.Module):
    """
    Lightweight convolutional VAE that encodes (B,3,H,W) → (B,c,H/8,W/8)
    and decodes back. Used to produce latents for the DiT training.
    """
    def __init__(self, latent_channels: int, image_size: int):
        super().__init__()
        c = latent_channels
        # Encoder: 3×H×W → c×(H/8)×(W/8)
        self.encoder = nn.Sequential(
            nn.Conv2d(3,   64,  4, 2, 1), nn.GroupNorm(8, 64),  nn.SiLU(),   # H/2
            nn.Conv2d(64,  128, 4, 2, 1), nn.GroupNorm(8, 128), nn.SiLU(),   # H/4
            nn.Conv2d(128, 256, 4, 2, 1), nn.GroupNorm(8, 256), nn.SiLU(),   # H/8
        )
        self.mu_conv  = nn.Conv2d(256, c, 1)
        self.lv_conv  = nn.Conv2d(256, c, 1)
        # Decoder: c×(H/8)×(W/8) → 3×H×W
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(c,   256, 4, 2, 1), nn.GroupNorm(8, 256), nn.SiLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.GroupNorm(8, 128), nn.SiLU(),
            nn.ConvTranspose2d(128,  64, 4, 2, 1), nn.GroupNorm(8, 64),  nn.SiLU(),
            nn.Conv2d(64, 3, 3, 1, 1), nn.Tanh(),
        )

    def encode(self, x):
        h  = self.encoder(x)
        mu = self.mu_conv(h)
        lv = self.lv_conv(h).clamp(-6, 6)
        return mu, lv

    def reparameterize(self, mu, lv):
        return mu + torch.randn_like(mu) * (0.5 * lv).exp()

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, lv = self.encode(x)
        z      = self.reparameterize(mu, lv)
        recon  = self.decode(z)
        recon_loss = F.mse_loss(recon, x)
        kl     = -0.5 * (1 + lv - mu.pow(2) - lv.exp()).mean()
        return recon, recon_loss + 0.001 * kl


def train_vae_encoder(latent_channels: int, n_epochs: int = None) -> ConvVAE:
    """Train a VAE encoder to produce meaningful latent representations."""
    if n_epochs is None:
        n_epochs = CFG.get('vae_epochs', 30)
    vae = ConvVAE(latent_channels, CFG['image_size']).to(DEVICE)
    opt = torch.optim.AdamW(vae.parameters(), lr=3e-4)
    total_batches = n_epochs * len(train_loader)
    sched_lr = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_batches)
    vae.train()
    pbar = tqdm(total=total_batches, desc=f'  VAE c={latent_channels}', leave=False)
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for batch in train_loader:
            imgs = batch[0].to(DEVICE)
            _, loss = vae(imgs)
            opt.zero_grad(); loss.backward(); opt.step()
            sched_lr.step()
            epoch_loss += loss.item()
            pbar.update(1)
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        avg = epoch_loss / len(train_loader)
        print(f'  VAE epoch {epoch+1}/{n_epochs}  avg_loss={avg:.4f}', flush=True)
    pbar.close()
    return vae


# ── Optional: use pretrained SD-VAE instead ────────────────────────────────────
# To use a pretrained VAE from HuggingFace (much better latents):
#
# pip install diffusers
# from diffusers.models import AutoencoderKL
# sd_vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-ema').to(DEVICE)
# sd_vae.requires_grad_(False)
#
# Then replace encode/decode calls below with:
#   z = sd_vae.encode(img).latent_dist.sample() * 0.18215
#   img_recon = sd_vae.decode(z / 0.18215).sample

print('ConvVAE defined. Will train one per latent channel count.')

---
## 7 · Training loop with mid-run evaluation

In [ ]:
from tqdm.auto import tqdm
from torch.amp import GradScaler, autocast


def prepare_vae_and_latents(latent_c: int):
    """Train or load VAE for one latent channel count and encode datasets.

    Run 1: trains VAE from scratch, encodes data, saves everything to CKPT_DIR.
    Runs 2+: loads saved VAE weights and pre-encoded latents from CKPT_DIR.
    """
    d_latent = latent_c * CFG['latent_spatial'] ** 2
    vae_file     = f'vae_c{latent_c}.pt'
    latents_file = f'latents_c{latent_c}.pt'

    # Check INPUT_CKPT_DIR (uploaded dataset) first, then local CKPT_DIR
    vae_path = latents_path = None
    for search_dir in filter(None, [INPUT_CKPT_DIR, CFG['ckpt_dir']]):
        v = os.path.join(search_dir, vae_file)
        l = os.path.join(search_dir, latents_file)
        if os.path.exists(v) and os.path.exists(l):
            vae_path, latents_path = v, l
            break

    if vae_path is not None and latents_path is not None:
        print(f'\n{"="*60}', flush=True)
        print(f'  Loading cached VAE + latents: c={latent_c}  (d_latent={d_latent})', flush=True)
        print(f'{"="*60}', flush=True)
        vae = ConvVAE(latent_c, CFG['image_size']).to(DEVICE)
        vae.load_state_dict(torch.load(vae_path, weights_only=True, map_location=DEVICE))
        vae.eval()
        cached = torch.load(latents_path, weights_only=False, map_location='cpu')
        Z_train  = cached['Z_train']
        Y_train  = cached['Y_train']
        Z_test   = cached['Z_test']
        Y_test   = cached['Y_test']
        eig_info = cached['eig_info']
        print(f'  Spectral gap: {eig_info["spectral_gap"]:.4f}  '
              f'Effective rank: {eig_info["effective_rank"]:.0f}', flush=True)
        return vae, Z_train, Y_train, Z_test, Y_test, eig_info

    print(f'\n{"="*60}', flush=True)
    print(f'  Training VAE: c={latent_c}  →  d_latent={d_latent}', flush=True)
    print(f'{"="*60}', flush=True)

    vae = train_vae_encoder(latent_c)
    vae.eval()

    @torch.no_grad()
    def encode_dataset(loader):
        latents, labels_list = [], []
        for imgs, lbls in loader:
            mu, _ = vae.encode(imgs.to(DEVICE))
            latents.append(mu.cpu())
            labels_list.append(lbls)
        return torch.cat(latents), torch.cat(labels_list)

    enc_loader_train = torch.utils.data.DataLoader(
        train_ds, batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers'], pin_memory=(DEVICE == 'cuda'),
        drop_last=False)
    enc_loader_test = torch.utils.data.DataLoader(
        test_ds, batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers'], pin_memory=(DEVICE == 'cuda'),
        drop_last=False)

    print('Encoding train set...', flush=True)
    Z_train, Y_train = encode_dataset(enc_loader_train)
    print('Encoding test set...', flush=True)
    Z_test, Y_test = encode_dataset(enc_loader_test)

    z_std = Z_train.std() + 1e-8
    Z_train, Z_test = Z_train / z_std, Z_test / z_std

    eig_info = eigenvalue_analysis(Z_train.numpy().reshape(len(Z_train), -1))
    print(f'  Spectral gap: {eig_info["spectral_gap"]:.4f}  '
          f'Effective rank: {eig_info["effective_rank"]:.0f}', flush=True)

    save_vae_path = os.path.join(CFG['ckpt_dir'], vae_file)
    save_lat_path = os.path.join(CFG['ckpt_dir'], latents_file)
    torch.save(vae.state_dict(), save_vae_path)
    torch.save({'Z_train': Z_train, 'Y_train': Y_train,
                'Z_test': Z_test, 'Y_test': Y_test,
                'eig_info': eig_info}, save_lat_path)
    print(f'  Saved VAE → {save_vae_path}', flush=True)
    print(f'  Saved latents → {save_lat_path}', flush=True)

    return vae, Z_train, Y_train, Z_test, Y_test, eig_info


def run_experiment(latent_c, n_train, Z_train_full, Y_train_full,
                   Z_test, Y_test, vae=None,
                   raw_incept_test=None, raw_incept_train=None,
                   real_feats_test=None):
    """Train DiT on n_train latents and evaluate."""
    d_latent = latent_c * CFG['latent_spatial'] ** 2

    # Per-run seed for reproducibility across configurations
    torch.manual_seed(CFG['seed'] + d_latent * 100 + n_train)
    np.random.seed(CFG['seed'] + d_latent * 100 + n_train)

    idx = torch.randperm(
        len(Z_train_full),
        generator=torch.Generator().manual_seed(CFG['seed'] + n_train),
    )[:n_train]
    Z_train = Z_train_full[idx]
    Y_train = Y_train_full[idx]

    print(f'\n  DiT: c={latent_c}, n_train={n_train}  (d_latent={d_latent})', flush=True)

    from torch.utils.data import TensorDataset as TDS
    latent_loader = torch.utils.data.DataLoader(
        TDS(Z_train, Y_train),
        batch_size=min(CFG['batch_size'], n_train),
        shuffle=True, pin_memory=(DEVICE == 'cuda'), drop_last=True,
    )

    model = build_dit(latent_c)
    ema   = EMA(model, decay=CFG['ema_decay'])
    opt   = torch.optim.AdamW(model.parameters(), lr=CFG['lr'],
                               betas=(0.9, 0.999), weight_decay=0.0)
    scaler = GradScaler('cuda', enabled=(CFG['mixed_prec'] and DEVICE=='cuda'))

    history = dict(
        steps=[],
        loss_steps=[], loss_vals=[],
        lfd_train=[], lfd_test=[],
        fid_test=[],
        memo_rate=[],
        flipd={t: {'mean': [], 'std': []} for t in CFG['flipd_t_vals']},
        gen_eff_rank=[],
        sample_imgs=[],
        tau_gen=None, tau_mem=None,
        tau_gen_metric=None,
    )

    real_feats_train = Z_train[:CFG['n_fid_samples']].numpy().reshape(
        min(CFG['n_fid_samples'], len(Z_train)), -1)
    if real_feats_test is None:
        real_feats_test = Z_test[:CFG['n_fid_samples']].numpy().reshape(
            min(CFG['n_fid_samples'], len(Z_test)), -1)

    # Fixed noise for sample progression (same noise across eval steps)
    n_prog = 16
    prog_shape = (n_prog, latent_c, CFG['latent_spatial'], CFG['latent_spatial'])
    prog_noise = torch.randn(prog_shape, device=DEVICE)
    prog_labels = (torch.arange(n_prog, device=DEVICE) % CFG['n_classes']
                   if CFG['n_classes'] > 0 else None)

    step = 0
    loader_iter = iter(latent_loader)
    pbar = tqdm(total=CFG['total_steps'],
                desc=f'c={latent_c} n={n_train}', leave=False)

    while step < CFG['total_steps']:
        model.train()
        try:
            z0_batch, lbl_batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(latent_loader)
            z0_batch, lbl_batch = next(loader_iter)

        z0    = z0_batch.to(DEVICE)
        lbls  = lbl_batch.to(DEVICE) if CFG['n_classes'] > 0 else None
        t     = torch.rand(z0.size(0), device=DEVICE)
        noise = torch.randn_like(z0)
        z_t, velocity = schedule.interpolate(z0, t, noise)

        with autocast('cuda', enabled=(CFG['mixed_prec'] and DEVICE=='cuda')):
            v_pred = model(z_t, t, lbls)
            loss   = F.mse_loss(v_pred, velocity)

        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(opt); scaler.update()
        ema.update(model)
        step += 1
        pbar.update(1)
        pbar.set_postfix(loss=f'{loss.item():.4f}')

        if step % CFG['loss_every'] == 0:
            history['loss_steps'].append(step)
            history['loss_vals'].append(loss.item())

        # ── Periodic evaluation ───────────────────────────────────────────────
        if step % CFG['eval_every'] == 0 or step == CFG['total_steps']:
            ema.shadow.eval()
            n_samp = CFG['n_fid_samples']
            shape  = (min(n_samp, 256), latent_c,
                      CFG['latent_spatial'], CFG['latent_spatial'])

            # Generate samples in batches
            fake_batches = []
            remaining = n_samp
            while remaining > 0:
                bs = min(256, remaining)
                s  = (bs, latent_c, CFG['latent_spatial'], CFG['latent_spatial'])
                lbl_s = (torch.randint(0, CFG['n_classes'], (bs,), device=DEVICE)
                         if CFG['n_classes'] > 0 else None)
                fake_batches.append(
                    euler_sample(ema.shadow, s,
                                 n_steps=CFG['fm_sample_steps'], labels=lbl_s).cpu()
                )
                remaining -= bs
            fake_z = torch.cat(fake_batches)[:n_samp]
            fake_feats = fake_z.numpy().reshape(len(fake_z), -1)

            lfd_tr = lfd_te = float('nan')
            try:
                lfd_tr = compute_fid(real_feats_train[:len(fake_feats)], fake_feats)
                lfd_te = compute_fid(real_feats_test[:len(fake_feats)], fake_feats)
            except Exception as e:
                print(f'  LFD error at step {step}: {e}', flush=True)

            # True FID via Inception-v3 (decode latents → pixels, compare
            # against raw real images so FID is comparable across channels)
            true_fid_te = float('nan')
            fake_incept = None
            if vae is not None and raw_incept_test is not None:
                try:
                    vae.eval()
                    with torch.no_grad():
                        decoded = []
                        for di in range(0, len(fake_z), 64):
                            decoded.append(vae.decode(
                                fake_z[di:di+64].to(DEVICE)).cpu())
                        fake_imgs = torch.cat(decoded)
                    fake_incept = inception_features(fake_imgs)
                    true_fid_te = compute_fid(raw_incept_test, fake_incept)
                except Exception as e:
                    print(f'  Inception FID error at step {step}: {e}',
                          flush=True)

            # NN memorisation (Carlini/Gu d1/d2 criterion) in Inception feature space
            if fake_incept is not None and raw_incept_train is not None:
                _, memo_rate = nn_memorization_ratio(
                    fake_incept, raw_incept_train,
                    threshold=CFG['memo_nn_thresh'])
            else:
                _, memo_rate = nn_memorization_ratio(
                    fake_feats, real_feats_train,
                    threshold=CFG['memo_nn_thresh'])

            # FLIPD via Hutchinson trace on DiT score network (multiple t values)
            n_flipd = min(32, len(Z_train))
            flipd_idx = torch.randperm(len(Z_train))[:n_flipd]
            flipd_z = Z_train[flipd_idx].to(DEVICE)
            flipd_y = (Y_train[flipd_idx].to(DEVICE)
                       if CFG['n_classes'] > 0 else None)
            for t_val in CFG['flipd_t_vals']:
                try:
                    lid_vals = flipd_lid_dit(
                        ema.shadow, flipd_z,
                        t_val=t_val,
                        n_probes=CFG['flipd_n_probes'],
                        labels=flipd_y)
                    history['flipd'][t_val]['mean'].append(float(lid_vals.mean().item()))
                    history['flipd'][t_val]['std'].append(float(lid_vals.std().item()))
                except Exception as e:
                    print(f'  FLIPD error at step {step}, t={t_val}: {e}')
                    history['flipd'][t_val]['mean'].append(float('nan'))
                    history['flipd'][t_val]['std'].append(float('nan'))

            # Eigenvalue analysis on generated samples
            gen_rank = float('nan')
            try:
                gen_eig = eigenvalue_analysis(fake_feats)
                gen_rank = gen_eig['effective_rank']
            except Exception as e:
                print(f'  Gen eigenvalue error at step {step}: {e}', flush=True)

            # Sample progression: generate from fixed noise, decode to pixels
            if vae is not None:
                try:
                    prog_z = euler_sample(
                        ema.shadow, prog_shape,
                        n_steps=CFG['fm_sample_steps'],
                        labels=prog_labels, x_init=prog_noise)
                    vae.eval()
                    with torch.no_grad():
                        prog_imgs = vae.decode(prog_z).cpu().clamp(0, 1)
                    history['sample_imgs'].append((step, prog_imgs))
                    grid = torchvision.utils.make_grid(prog_imgs, nrow=4, padding=2)
                    grid_path = os.path.join(
                        CFG['ckpt_dir'],
                        f'samples_c{latent_c}_n{n_train}_step{step}.png')
                    torchvision.utils.save_image(grid, grid_path)
                except Exception as e:
                    print(f'  Sample grid error at step {step}: {e}', flush=True)

            history['steps'].append(step)
            history['lfd_train'].append(lfd_tr)
            history['lfd_test'].append(lfd_te)
            history['fid_test'].append(true_fid_te)
            history['memo_rate'].append(memo_rate)
            history['gen_eff_rank'].append(gen_rank)

            # 1. Determine the metric for tau_gen (with a warning if the fallback is used)
            if not np.isnan(true_fid_te):
                fid_for_tau = true_fid_te
            else:
                fid_for_tau = lfd_te
                print(f"  [Step {step}] True FID is NaN. Falling back to LFD ({lfd_te:.2f}) for tau_gen check.", flush=True)

            # 2. Check for Generation Onset (tau_gen)
            if history['tau_gen'] is None and fid_for_tau < CFG['fid_threshold']:
                history['tau_gen'] = step
                history['tau_gen_metric'] = 'FID' if not np.isnan(true_fid_te) else 'LFD'
                print(f"  [Step {step}] Generation onset (tau_gen) reached via {history['tau_gen_metric']}! "
                      f"(Score: {fid_for_tau:.2f} < {CFG['fid_threshold']})", flush=True)

            # 3. Check for Memorization Onset (tau_mem)
            if history['tau_mem'] is None and memo_rate > CFG['memo_threshold']:
                history['tau_mem'] = step
                print(f"  [Step {step}] Memorization onset (tau_mem) reached! (Rate: {memo_rate:.3f} > {CFG['memo_threshold']})", flush=True)

            _flipd_str = '  '.join(
                f't{t}={history["flipd"][t]["mean"][-1]:.1f}'
                for t in CFG['flipd_t_vals'])
            print(f'  step={step:6d}  FID={true_fid_te:.1f}  '
                  f'LFD={lfd_te:.1f}  memo={memo_rate:.3f}  '
                  f'FLIPD[{_flipd_str}]  '
                  f'gen_rank={gen_rank:.0f}', flush=True)

            if CFG['save_ckpt'] and step == CFG['total_steps']:
                ckpt_path = os.path.join(
                    CFG['ckpt_dir'],
                    f'dit_c{latent_c}_n{n_train}_step{step}.pt')
                torch.save({'model': ema.shadow.state_dict(),
                            'step': step, 'history': history}, ckpt_path)

    pbar.close()
    print(f'  τ_gen={history["tau_gen"]}  τ_mem={history["tau_mem"]}', flush=True)
    return history


print('Training functions defined.')

---
## 8 · Run the latent dim sweep

In [ ]:
# ── Raw test images + Inception features (cached to disk) ────────────────────
raw_incept_path = os.path.join(CFG['ckpt_dir'], 'raw_incept_test.npy')
print('Collecting raw test images...', flush=True)
raw_test_imgs = []
for batch in test_loader:
    raw_test_imgs.append(batch[0])
    if sum(b.shape[0] for b in raw_test_imgs) >= CFG['n_fid_samples']:
        break
raw_test_imgs = torch.cat(raw_test_imgs)[:CFG['n_fid_samples']]
print(f'  Raw test images: {raw_test_imgs.shape}', flush=True)

# Check for cached features (from INPUT_CKPT_DIR or local CKPT_DIR)
_cached_incept = None
for _dir in filter(None, [INPUT_CKPT_DIR, CFG['ckpt_dir']]):
    _p = os.path.join(_dir, 'raw_incept_test.npy')
    if os.path.exists(_p):
        _cached_incept = _p
        break

if _cached_incept is not None:
    raw_incept_test = np.load(_cached_incept)
    print(f'  Loaded cached Inception features: {raw_incept_test.shape}', flush=True)
else:
    print('Extracting Inception features for raw test images...', flush=True)
    raw_incept_test = inception_features(raw_test_imgs)
    np.save(raw_incept_path, raw_incept_test)
    print(f'  Raw Inception features: {raw_incept_test.shape} (saved)', flush=True)

# ── Raw train images + Inception features (cached to disk) ───────────────────
print('Collecting raw train images...', flush=True)
raw_train_imgs = []
for batch in train_loader:
    raw_train_imgs.append(batch[0])
    if sum(b.shape[0] for b in raw_train_imgs) >= CFG['n_fid_samples']:
        break
raw_train_imgs = torch.cat(raw_train_imgs)[:CFG['n_fid_samples']]
print(f'  Raw train images: {raw_train_imgs.shape}', flush=True)

_cached_incept_train = None
for _dir in filter(None, [INPUT_CKPT_DIR, CFG['ckpt_dir']]):
    _p = os.path.join(_dir, 'raw_incept_train.npy')
    if os.path.exists(_p):
        _cached_incept_train = _p
        break

if _cached_incept_train is not None:
    raw_incept_train = np.load(_cached_incept_train)
    print(f'  Loaded cached train Inception features: {raw_incept_train.shape}', flush=True)
else:
    print('Extracting Inception features for raw train images...', flush=True)
    raw_incept_train = inception_features(raw_train_imgs)
    np.save(os.path.join(CFG['ckpt_dir'], 'raw_incept_train.npy'), raw_incept_train)
    print(f'  Train Inception features: {raw_incept_train.shape} (saved)', flush=True)

# ── Outer sweep: (latent channels) × (dataset sizes for this RUN_ID) ─────────
all_histories = {}
vae_cache = {}
eig_cache = {}
vae_baseline_cache = {}

for c in CFG['latent_channels']:
    vae, Z_tr, Y_tr, Z_te, Y_te, eig_info = prepare_vae_and_latents(c)
    vae_cache[c] = vae
    eig_cache[c] = eig_info

    # Precompute LFD test reference (same for all n_train within this c)
    real_feats_test_c = Z_te[:CFG['n_fid_samples']].numpy().reshape(
        min(CFG['n_fid_samples'], len(Z_te)), -1)

    # VAE baseline FID: raw real vs VAE-reconstructed real (encode→decode).
    # Shows the quality ceiling imposed by VAE compression at this channel count.
    recon_incept_file = f'recon_incept_c{c}.npy'
    _cached_recon = None
    for _dir in filter(None, [INPUT_CKPT_DIR, CFG['ckpt_dir']]):
        _p = os.path.join(_dir, recon_incept_file)
        if os.path.exists(_p):
            _cached_recon = _p
            break

    if _cached_recon is not None:
        recon_incept = np.load(_cached_recon)
        print(f'  Loaded cached recon Inception features for c={c}: {recon_incept.shape}', flush=True)
    else:
        print(f'  Computing VAE baseline FID for c={c}...', flush=True)
        vae.eval()
        with torch.no_grad():
            recon_imgs = []
            for ri in range(0, len(raw_test_imgs), 64):
                batch = raw_test_imgs[ri:ri+64].to(DEVICE)
                mu, _ = vae.encode(batch)
                recon_imgs.append(vae.decode(mu).cpu())
            recon_test = torch.cat(recon_imgs)
        recon_incept = inception_features(recon_test)
        np.save(os.path.join(CFG['ckpt_dir'], recon_incept_file), recon_incept)
        print(f'  Saved recon Inception features for c={c}: {recon_incept.shape}', flush=True)

    vae_baseline_fid = compute_fid(raw_incept_test, recon_incept)
    vae_baseline_cache[c] = vae_baseline_fid
    print(f'  VAE reconstruction baseline FID: {vae_baseline_fid:.1f}', flush=True)

    for n in CFG['n_train_grid']:
        n_actual = min(n, len(Z_tr))
        if n_actual < n:
            print(f'  Clamping n_train={n} → {n_actual} (available)', flush=True)
        hist = run_experiment(c, n_actual, Z_tr, Y_tr, Z_te, Y_te,
                              vae=vae, raw_incept_test=raw_incept_test,
                              raw_incept_train=raw_incept_train,
                              real_feats_test=real_feats_test_c)
        all_histories[(c, n)] = hist

# ── Save this run's results ──────────────────────────────────────────────────
run_save_path = os.path.join(OUT_DIR, f'results_run{RUN_ID}.pt')
_save_histories = {}
for (c, n), h in all_histories.items():
    _h = {k: v for k, v in h.items() if k != 'sample_imgs'}
    _save_histories[f'{c}_{n}'] = _h
torch.save({
    'all_histories': _save_histories,
    'eig_cache': eig_cache,
    'vae_baseline_cache': vae_baseline_cache,
    'run_id': RUN_ID,
    'n_train_grid': CFG['n_train_grid'],
}, run_save_path)
print(f'\nRun {RUN_ID} results saved → {run_save_path}')

_flipd_hdr = '  '.join(f'FLIPD_{t}' for t in CFG['flipd_t_vals'])
print(f'\n{"c":>4}  {"d_lat":>6}  {"n":>6}  {"tau_gen":>8}  '
      f'{"tau_mem":>8}  {"FID_te":>8}  {"LFD_te":>8}  {"memo":>6}  {_flipd_hdr}')
for (c, n), h in all_histories.items():
    d = c * CFG['latent_spatial']**2
    best_fid = min(h['fid_test']) if h['fid_test'] else float('nan')
    best_lfd = min(h['lfd_test']) if h['lfd_test'] else float('nan')
    final_memo = h['memo_rate'][-1] if h['memo_rate'] else float('nan')
    flipd_strs = []
    for t_val in CFG['flipd_t_vals']:
        fdata = h.get('flipd', {}).get(t_val, {}).get('mean', [])
        flipd_strs.append(f'{fdata[-1]:>8.1f}' if fdata else f'{"nan":>8}')
    print(f'{c:>4}  {d:>6}  {n:>6}  '
          f'{str(h["tau_gen"]):>8}  {str(h["tau_mem"]):>8}  '
          f'{best_fid:>8.1f}  {best_lfd:>8.1f}  '
          f'{final_memo:>6.3f}  {"  ".join(flipd_strs)}')

---
## 8b · Merge results from all runs
> Run this cell **after all 3 Kaggle sessions** are complete.
> Upload `results_run1.pt`, `results_run2.pt`, `results_run3.pt` from each session's output
> into `OUT_DIR` (or `/kaggle/working/track2/`), then execute.

In [ ]:
# ── Merge results from all runs ───────────────────────────────────────────────
import glob as _glob

merged_histories = {}
merged_eig_cache = {}
merged_vae_baseline = {}

run_files = sorted(_glob.glob(os.path.join(OUT_DIR, 'results_run*.pt')))
print(f'Found {len(run_files)} result file(s): {[os.path.basename(f) for f in run_files]}')

for rf in run_files:
    data = torch.load(rf, weights_only=False, map_location='cpu')
    for key_str, hist in data['all_histories'].items():
        c, n = int(key_str.split('_')[0]), int(key_str.split('_')[1])
        # Backwards compat: old runs used fid_train/fid_test for latent FD
        if 'lfd_test' not in hist and 'fid_test' in hist:
            hist['lfd_train'] = hist.pop('fid_train', [])
            hist['lfd_test']  = hist.pop('fid_test', [])
            hist['fid_test']  = [float('nan')] * len(hist['lfd_test'])
        # Backwards compat: old runs stored flipd as flat lists
        if 'flipd_mean' in hist and 'flipd' not in hist:
            default_t = CFG['flipd_primary_t']
            hist['flipd'] = {default_t: {'mean': hist.pop('flipd_mean'),
                                          'std': hist.pop('flipd_std')}}
        merged_histories[(c, n)] = hist
    for c_str, eig in data['eig_cache'].items():
        merged_eig_cache[int(c_str) if isinstance(c_str, str) else c_str] = eig
    for c_key, fid_val in data.get('vae_baseline_cache', {}).items():
        merged_vae_baseline[int(c_key) if isinstance(c_key, str) else c_key] = fid_val
    print(f'  Loaded run {data["run_id"]}: n_train={data["n_train_grid"]}')

all_histories = merged_histories
eig_cache = merged_eig_cache
vae_baseline_cache = merged_vae_baseline

print(f'\nMerged grid: {len(all_histories)} experiments')
for (c, n) in sorted(all_histories.keys()):
    d = c * CFG['latent_spatial']**2
    h = all_histories[(c, n)]
    best_fid = min(h['fid_test']) if h['fid_test'] else float('nan')
    print(f'  c={c:>2}  d={d:>4}  n={n:>5}  best_FID={best_fid:.1f}  '
          f'tau_gen={h.get("tau_gen")}  tau_mem={h.get("tau_mem")}')

---
## 9 · Results — full visualisation dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10})

cs       = CFG['latent_channels']
ns       = sorted(set(n for (_, n) in all_histories.keys()))
n_max    = max(ns) if ns else 0
d_lats   = [c * CFG['latent_spatial']**2 for c in cs]
cmap_c   = plt.cm.plasma(np.linspace(0.1, 0.9, len(cs)))


def H(c, n=None):
    """Access history for (c, n); defaults to largest n_train."""
    if n is None:
        n = n_max
    h = all_histories.get((c, n))
    if h:
        return h
    return dict(steps=[],
                loss_steps=[], loss_vals=[],
                lfd_train=[], lfd_test=[], fid_test=[],
                memo_rate=[],
                flipd={t: {'mean': [], 'std': []} for t in CFG['flipd_t_vals']},
                gen_eff_rank=[], sample_imgs=[],
                tau_gen=None, tau_mem=None,
                tau_gen_metric=None)


fig = plt.figure(figsize=(20, 23))
gs  = gridspec.GridSpec(5, 3, hspace=0.50, wspace=0.38,
                        left=0.06, right=0.97, top=0.96, bottom=0.03)

# ── (a) True FID (Inception) vs steps ─────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
for i, c in enumerate(cs):
    h = H(c)
    if not h['fid_test']: continue
    ax.plot(h['steps'], h['fid_test'], color=cmap_c[i], lw=2,
            label=f'c={c} (d={d_lats[i]})')
    if h['tau_gen']:
        metric_tag = h.get('tau_gen_metric', '?')
        ax.axvline(h['tau_gen'], color=cmap_c[i], ls=':', lw=1, alpha=0.7)
        ax.annotate(f'τ_gen({metric_tag})', xy=(h['tau_gen'], 0.97),
                    xycoords=('data', 'axes fraction'), fontsize=6,
                    color=cmap_c[i], rotation=90, ha='right', va='top')
    bl = vae_baseline_cache.get(c)
    if bl is not None:
        ax.axhline(bl, color=cmap_c[i], ls='--', lw=1, alpha=0.4)
ax.axhline(CFG['fid_threshold'], color='#6b7280', ls='--', lw=1.5,
           label=f'τ_gen thresh ({CFG["fid_threshold"]})')
ax.set_title(f'(a) Inception FID vs steps  (n={n_max})\n'
             f'(dashed lines = VAE reconstruction baseline per c)', fontweight='bold',
             fontsize=9)
ax.set_xlabel('Training steps'); ax.set_ylabel('FID (Inception)')
ax.legend(fontsize=7, ncol=1); ax.grid(alpha=0.2)

# ── (b) LFD (latent Fréchet distance) vs steps ───────────────────────────────
ax = fig.add_subplot(gs[0, 1])
for i, c in enumerate(cs):
    h = H(c)
    if not h['lfd_test']: continue
    ax.plot(h['steps'], h['lfd_test'], color=cmap_c[i], lw=2,
            label=f'c={c} (d={d_lats[i]})')
ax.set_title(f'(b) Latent FD vs steps  (n={n_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('LFD (latent space)')
ax.legend(fontsize=7, ncol=1); ax.grid(alpha=0.2)

# ── (c) NN memorisation rate over training steps ─────────────────────────────
ax = fig.add_subplot(gs[0, 2])
for i, c in enumerate(cs):
    h = H(c)
    if not h['memo_rate']: continue
    ax.plot(h['steps'], h['memo_rate'], color=cmap_c[i], lw=2, label=f'c={c}')
    if h['tau_mem']:
        ax.axvline(h['tau_mem'], color=cmap_c[i], ls=':', lw=1, alpha=0.7)
ax.axhline(CFG['memo_threshold'], color='#dc2626', ls='--', lw=1.5,
           label=f'τ_mem thresh ({CFG["memo_threshold"]})')
ax.set_title(f'(c) NN memo rate vs steps  (n={n_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('Memo rate (d₁/d₂ < thresh)')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (d) Eigenvalue spectra of VAE latent space ───────────────────────────────
ax = fig.add_subplot(gs[1, 0])
for i, c in enumerate(cs):
    eig = eig_cache.get(c)
    if eig is None: continue
    eigvals = eig['eigenvalues']
    n_show = min(50, len(eigvals))
    ax.semilogy(range(1, n_show + 1), eigvals[:n_show],
                color=cmap_c[i], lw=1.8, marker='.', ms=4,
                label=f'c={c} (rank={eig["effective_rank"]:.0f})')
    gap_pos = eig['gap_position']
    if 0 < gap_pos < n_show:
        ax.axvline(gap_pos + 1, color=cmap_c[i], ls=':', lw=1, alpha=0.6)
ax.set_title('(d) Eigenvalue spectra of latent covariance', fontweight='bold')
ax.set_xlabel('Eigenvalue index'); ax.set_ylabel('Eigenvalue (log scale)')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (e) τ_gen and τ_mem vs d_latent ──────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
tau_gen_vals = [H(c).get('tau_gen') or CFG['total_steps'] for c in cs]
tau_mem_vals = [H(c).get('tau_mem') or CFG['total_steps'] for c in cs]
tau_gen_metrics = [H(c).get('tau_gen_metric') for c in cs]
ax.plot(d_lats, tau_gen_vals, 'o-', color='#2563eb', lw=2, ms=7, label='τ_gen')
ax.plot(d_lats, tau_mem_vals, 's-', color='#dc2626', lw=2, ms=7, label='τ_mem')
for d, tg_val, tg_metric in zip(d_lats, tau_gen_vals, tau_gen_metrics):
    if tg_metric is not None:
        ax.annotate(tg_metric, (d, tg_val), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=7, color='#2563eb',
                    fontweight='bold')
ax.fill_between(d_lats, tau_gen_vals, tau_mem_vals,
                alpha=0.10, color='#dc2626', label='safe window')
ax.set_title(f'(e) τ_gen and τ_mem vs d_latent  (n={n_max})', fontweight='bold')
ax.set_xlabel('d_latent'); ax.set_ylabel('Training steps at onset')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# ── (f) Best FID & LFD vs d_latent ───────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
best_fids  = [min(H(c)['fid_test']) if H(c)['fid_test'] else float('nan') for c in cs]
best_lfds  = [min(H(c)['lfd_test']) if H(c)['lfd_test'] else float('nan') for c in cs]
vae_bl_vals = [vae_baseline_cache.get(c, float('nan')) for c in cs]
ax.plot(d_lats, best_fids, 'o-', color='#7c3aed', lw=2, ms=7, label='Best FID (Inception)')
ax.plot(d_lats, vae_bl_vals, 'D--', color='#dc2626', lw=1.5, ms=5, alpha=0.7, label='VAE baseline FID')
ax.plot(d_lats, best_lfds, 'x--', color='#7c3aed', lw=1, ms=6, alpha=0.5, label='Best LFD (latent)')
ax2_twin = ax.twinx()
_flipd_markers = ['s', '^', 'v', 'D']
for _ti, t_val in enumerate(CFG['flipd_t_vals']):
    _flipd_last_t = []
    for c in cs:
        fdata = H(c).get('flipd', {}).get(t_val, {}).get('mean', [])
        _flipd_last_t.append(fdata[-1] if fdata else float('nan'))
    ax2_twin.plot(d_lats, _flipd_last_t,
                  f'{_flipd_markers[_ti % len(_flipd_markers)]}--',
                  color='#d97706', lw=1.2, ms=5,
                  alpha=0.5 + 0.5 * (t_val == CFG['flipd_primary_t']),
                  label=f'FLIPD t={t_val}')
ax2_twin.set_ylabel('FLIPD estimate', color='#d97706')
ax2_twin.tick_params(axis='y', labelcolor='#d97706')
ax.set_title('(f) Best FID, LFD & FLIPD vs d_latent', fontweight='bold')
ax.set_xlabel('d_latent'); ax.set_ylabel('FID / LFD', color='#7c3aed')
ax.tick_params(axis='y', labelcolor='#7c3aed')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=7); ax.grid(alpha=0.2)

# ── (g) Diffusion loss curves ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
for i, c in enumerate(cs):
    h = H(c)
    l_steps = h.get('loss_steps', [])
    l_vals  = h.get('loss_vals', [])
    if not l_vals: continue
    ax.plot(l_steps, l_vals, color=cmap_c[i], lw=1.5, alpha=0.85, label=f'c={c}')
ax.set_title(f'(g) Flow matching loss vs steps  (n={n_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('MSE loss')
ax.legend(fontsize=7.5, ncol=2); ax.grid(alpha=0.2)

# ── (h) FLIPD trajectory (multi-t) ────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 1])
_flipd_ls = {t: ls for t, ls in zip(CFG['flipd_t_vals'],
                                      ['-', '--', '-.', ':'])}
for i, c in enumerate(cs):
    h = H(c)
    for t_val, ls in _flipd_ls.items():
        fdata = h.get('flipd', {}).get(t_val, {})
        fmean = np.array(fdata.get('mean', []))
        if not len(fmean): continue
        ax.plot(h['steps'], fmean, color=cmap_c[i], ls=ls, lw=1.4,
                label=f'c={c} t={t_val}')
ax.set_title(f'(h) FLIPD LID vs steps  (n={n_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('FLIPD LID estimate')
ax.legend(fontsize=6, ncol=3); ax.grid(alpha=0.2)

# ── (i) τ_mem − τ_gen gap vs d_latent ────────────────────────────────────────
ax = fig.add_subplot(gs[2, 2])
raw_tg = [H(c).get('tau_gen') for c in cs]
raw_tm = [H(c).get('tau_mem') for c in cs]
raw_tg_metrics = [H(c).get('tau_gen_metric') for c in cs]
gaps = []
bar_cols = []
bar_labels = []
for tg, tm, tg_m in zip(raw_tg, raw_tm, raw_tg_metrics):
    if tg is None or tm is None:
        gaps.append(0)
        bar_cols.append('#9ca3af')
        bar_labels.append('N/A')
    else:
        g = tm - tg
        gaps.append(g)
        bar_cols.append('#22c55e' if g > 0 else '#ef4444')
        metric_suffix = f' ({tg_m})' if tg_m else ''
        bar_labels.append(f'{g:+.0f}{metric_suffix}')
bars = ax.bar(d_lats, gaps, color=bar_cols, alpha=0.85, edgecolor='white', linewidth=0.8)
max_gap = max((abs(g) for g in gaps), default=1)
for bar, lbl in zip(bars, bar_labels):
    y_pos = bar.get_height() + max_gap * 0.02 if bar.get_height() >= 0 else bar.get_height() - max_gap * 0.06
    ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
            lbl, ha='center', va='bottom', fontsize=8,
            fontstyle='italic' if lbl == 'N/A' else 'normal')
ax.axhline(0, color='#374151', lw=1)
ax.set_title('(i) τ_mem − τ_gen  (positive = quality before memorisation)',
             fontweight='bold')
ax.set_xlabel('d_latent'); ax.set_ylabel('Steps gap')
ax.grid(axis='y', alpha=0.2)

# ── (j) Memorisation heatmap (d_latent × n_train) ────────────────────────────
ax = fig.add_subplot(gs[3, 0])
memo_grid = np.full((max(len(ns), 1), len(cs)), float('nan'))
for j, c in enumerate(cs):
    for i_n, n in enumerate(ns):
        h = all_histories.get((c, n))
        if h and h['memo_rate']:
            memo_grid[i_n, j] = h['memo_rate'][-1]
memo_max = np.nanmax(memo_grid) if memo_grid.size and not np.all(np.isnan(memo_grid)) else 0.2
im = ax.imshow(memo_grid, aspect='auto', origin='lower', cmap='RdYlGn_r',
               vmin=0, vmax=max(0.2, memo_max))
ax.set_xticks(range(len(cs))); ax.set_xticklabels(d_lats)
ax.set_yticks(range(len(ns))); ax.set_yticklabels(ns)
ax.set_xlabel('d_latent'); ax.set_ylabel('n_train')
ax.set_title('(j) Final memo rate  (d_latent × n_train)', fontweight='bold')
fig.colorbar(im, ax=ax, label='Memo rate', shrink=0.85)

# ── (k) Hypothesis test: d* vs d_intrinsic ───────────────────────────────────
ax = fig.add_subplot(gs[3, 1])
valid_fids = [(d, f) for d, f in zip(d_lats, best_fids) if not np.isnan(f)]
if not valid_fids:
    valid_fids = [(d, f) for d, f in zip(d_lats, best_lfds) if not np.isnan(f)]
d_star = min(valid_fids, key=lambda x: x[1])[0] if valid_fids else d_lats[0]
largest_c = max(cs)
h_largest = H(largest_c)
_pt = CFG['flipd_primary_t']
_fdata = h_largest.get('flipd', {}).get(_pt, {}).get('mean', [])
if _fdata and not np.isnan(_fdata[-1]):
    d_intrinsic = round(_fdata[-1])
    flipd_source = f'c={largest_c}, t={_pt}'
else:
    _all_flipd = []
    for c in cs:
        fd = H(c).get('flipd', {}).get(_pt, {}).get('mean', [])
        if fd and not np.isnan(fd[-1]):
            _all_flipd.append(fd[-1])
    d_intrinsic = round(max(_all_flipd)) if _all_flipd else 0
    flipd_source = f'max fallback, t={_pt}'
bar_data = {'Best d_latent\n(min FID)': d_star,
            f'FLIPD d_int\n({flipd_source})': d_intrinsic}
bars = ax.bar(list(bar_data.keys()), list(bar_data.values()),
              color=['#2563eb', '#d97706'], alpha=0.85, edgecolor='white', width=0.5)
for bar, v in zip(bars, bar_data.values()):
    ax.text(bar.get_x()+bar.get_width()/2, v + 0.2,
            str(v), ha='center', va='bottom', fontsize=11, fontweight='bold')
rel_err = abs(d_star - d_intrinsic) / (d_intrinsic + 1e-8) * 100
ax.set_title(f'(k) Hypothesis: d* ≈ d_int\nRelative error: {rel_err:.1f}%',
             fontweight='bold')
ax.set_ylabel('Dimension'); ax.grid(axis='y', alpha=0.2)
ax.set_ylim(0, max(d_star, d_intrinsic, 1) * 1.4)

# ── (l) Effective rank vs d_latent ────────────────────────────────────────────
ax = fig.add_subplot(gs[3, 2])
eff_ranks = [eig_cache.get(c, {}).get('effective_rank', float('nan')) for c in cs]
spec_gaps = [eig_cache.get(c, {}).get('spectral_gap', float('nan')) for c in cs]
ax.bar(d_lats, eff_ranks, color='#0ea5e9', alpha=0.8, edgecolor='white',
       label='Effective rank')
ax2_twin = ax.twinx()
ax2_twin.plot(d_lats, spec_gaps, 's-', color='#e11d48', lw=2, ms=7,
              label='Spectral gap')
ax2_twin.set_ylabel('Spectral gap', color='#e11d48')
ax2_twin.tick_params(axis='y', labelcolor='#e11d48')
ax.set_title('(l) Effective rank & spectral gap vs d_latent', fontweight='bold')
ax.set_xlabel('d_latent'); ax.set_ylabel('Effective rank', color='#0ea5e9')
ax.tick_params(axis='y', labelcolor='#0ea5e9')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8); ax.grid(alpha=0.2)

# ── (m) Generated effective rank vs steps ──────────────────────────────────────
ax = fig.add_subplot(gs[4, 0])
for i, c in enumerate(cs):
    h = H(c)
    ranks = h.get('gen_eff_rank', [])
    if not ranks: continue
    ax.plot(h['steps'], ranks, color=cmap_c[i], lw=1.8, label=f'c={c}')
    train_rank = eig_cache.get(c, {}).get('effective_rank')
    if train_rank is not None:
        ax.axhline(train_rank, color=cmap_c[i], ls=':', lw=1, alpha=0.5)
ax.set_title(f'(m) Generated effective rank vs steps  (n={n_max})', fontweight='bold')
ax.set_xlabel('Training steps'); ax.set_ylabel('Effective rank')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── (n) Final gen rank vs train rank by d_latent ─────────────────────────────
ax = fig.add_subplot(gs[4, 1])
train_ranks = [eig_cache.get(c, {}).get('effective_rank', float('nan')) for c in cs]
final_gen_ranks = []
for c in cs:
    h = H(c)
    ranks = h.get('gen_eff_rank', [])
    final_gen_ranks.append(ranks[-1] if ranks else float('nan'))
x_pos = np.arange(len(cs))
w = 0.35
ax.bar(x_pos - w/2, train_ranks, w, color='#0ea5e9', alpha=0.8, label='Train rank')
ax.bar(x_pos + w/2, final_gen_ranks, w, color='#f97316', alpha=0.8, label='Gen rank (final)')
ax.set_xticks(x_pos); ax.set_xticklabels(d_lats)
ax.set_xlabel('d_latent'); ax.set_ylabel('Effective rank')
ax.set_title('(n) Train vs generated effective rank', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.2)

# ── (o) Sample progression ────────────────────────────────────────────────────
ax = fig.add_subplot(gs[4, 2])
_prog_c = max(cs)
_prog_imgs = H(_prog_c).get('sample_imgs', [])
if _prog_imgs:
    n_show = min(5, len(_prog_imgs))
    indices = np.linspace(0, len(_prog_imgs) - 1, n_show, dtype=int)
    cols_per_step = 4
    composite = []
    step_labels = []
    for idx in indices:
        s, imgs = _prog_imgs[idx]
        composite.append(imgs[:cols_per_step])
        step_labels.append(f'{s//1000}k' if s >= 1000 else str(s))
    composite = torch.cat(composite, dim=0)
    grid_img = torchvision.utils.make_grid(
        composite, nrow=cols_per_step, padding=1, pad_value=1.0)
    ax.imshow(grid_img.permute(1, 2, 0).numpy())
    for ri, lbl in enumerate(step_labels):
        y_pos = ri * (imgs.shape[-1] + 1) + imgs.shape[-1] // 2
        ax.text(-2, y_pos + 1, lbl, ha='right', va='center', fontsize=7,
                fontweight='bold')
    ax.set_title(f'(o) Sample progression  (c={_prog_c})', fontweight='bold')
ax.axis('off')

fig.suptitle(
    f'Track 2 — DiT-{CFG["dit_variant"]} | {CFG["dataset"]} | '
    f'flow matching | {CFG["total_steps"]:,} steps',
    fontsize=14, fontweight='bold', y=0.97
)

dash_path = os.path.join(OUT_DIR, 'track2_dashboard.png')
plt.savefig(dash_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved → {dash_path}')

---
## 10 · Styled summary table

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for (c, n), h in all_histories.items():
    d = c * CFG['latent_spatial']**2
    best_fid   = min(h['fid_test']) if h['fid_test'] else float('nan')
    best_lfd   = min(h.get('lfd_test', [float('nan')])) if h.get('lfd_test') else float('nan')
    final_memo = h['memo_rate'][-1] if h['memo_rate'] else float('nan')

    if   final_memo > CFG['memo_threshold']:      memo_flag = '🔴 memorising'
    elif final_memo > CFG['memo_threshold'] / 2:  memo_flag = '🟡 borderline'
    else:                                          memo_flag = '🟢 generalising'

    vae_bl = vae_baseline_cache.get(c, float('nan'))

    row = dict(
        c             = c,
        d_latent      = d,
        n_train       = n,
        VAE_baseline  = round(vae_bl, 2),
        best_FID_test = round(best_fid, 2),
        best_LFD_test = round(best_lfd, 2),
        tau_gen       = h.get('tau_gen'),
        tau_gen_metric= h.get('tau_gen_metric', 'N/A'),
        tau_mem       = h.get('tau_mem'),
        tau_gap       = (h['tau_mem'] - h['tau_gen']
                         if h.get('tau_mem') is not None and h.get('tau_gen') is not None
                         else None),
        final_memo    = round(final_memo, 3),
        memo_status   = memo_flag,
        gen_eff_rank  = round(h['gen_eff_rank'][-1], 1) if h.get('gen_eff_rank') else float('nan'),
    )
    for t_val in CFG['flipd_t_vals']:
        fdata = h.get('flipd', {}).get(t_val, {}).get('mean', [])
        row[f'FLIPD_t{t_val}'] = round(fdata[-1], 2) if fdata else float('nan')
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, 'track2_results.csv')
df.to_csv(csv_path, index=False)
print(f'CSV saved → {csv_path}\n')

def col_memo(v):
    if v > CFG['memo_threshold']:       return 'background-color:#fee2e2;color:#991b1b'
    if v > CFG['memo_threshold'] / 2:   return 'background-color:#fef3c7;color:#92400e'
    return 'background-color:#dcfce7;color:#166534'

def col_gap(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return 'background-color:#f3f4f6;color:#6b7280'
    if v > 5000:  return 'background-color:#dcfce7;color:#166534'
    if v > 0:     return 'background-color:#fef3c7;color:#92400e'
    return 'background-color:#fee2e2;color:#991b1b'

def col_fid(v):
    if v < 30:  return 'background-color:#dcfce7;color:#166534'
    if v < 80:  return 'background-color:#fef3c7;color:#92400e'
    return 'background-color:#fee2e2;color:#991b1b'

styled = (
    df.style
      .map(col_memo, subset=['final_memo'])
      .map(col_gap,  subset=['tau_gap'])
      .map(col_fid,  subset=['best_FID_test'])
      .format({**{'VAE_baseline': '{:.2f}', 'best_FID_test': '{:.2f}',
                  'best_LFD_test': '{:.2f}', 'final_memo': '{:.3f}'},
               **{f'FLIPD_t{t}': '{:.2f}' for t in CFG['flipd_t_vals']}})
      .set_caption(
          f'Track 2 results — DiT-{CFG["dit_variant"]} on {CFG["dataset"]} '
          f'(flow matching, {CFG["total_steps"]:,} steps)')
      .set_table_styles([{'selector':'caption',
                          'props':[('font-size','13px'),('font-weight','bold'),
                                   ('padding-bottom','8px')]}])
)
display(styled)

# ── Final interpretation ──────────────────────────────────────────────────────
ns_avail = sorted(set(n for (_, n) in all_histories.keys()))
n_max_s = max(ns_avail) if ns_avail else 0

def _best_fid(c):
    h = all_histories.get((c, n_max_s), {})
    fids = h.get('fid_test', [9999])
    fids = [f for f in fids if not np.isnan(f)]
    return min(fids) if fids else 9999

best_c = min(cs, key=_best_fid)
best_d = best_c * CFG['latent_spatial']**2
largest_c = max(cs)
h_largest_interp = all_histories.get((largest_c, n_max_s), {})
_pt = CFG['flipd_primary_t']
_fdata_interp = h_largest_interp.get('flipd', {}).get(_pt, {}).get('mean', [])
if _fdata_interp and not np.isnan(_fdata_interp[-1]):
    d_int = round(_fdata_interp[-1], 2)
    flipd_source_interp = f'from c={largest_c}, t={_pt}'
else:
    _all_flipd_interp = []
    for c in cs:
        fd = all_histories.get((c, n_max_s), {}).get('flipd', {}).get(_pt, {}).get('mean', [])
        if fd and not np.isnan(fd[-1]):
            _all_flipd_interp.append(fd[-1])
    d_int = round(max(_all_flipd_interp), 2) if _all_flipd_interp else 0
    flipd_source_interp = f'max fallback, t={_pt}'

tau_gen_metrics_used = set(
    h.get('tau_gen_metric') for h in all_histories.values()
    if h.get('tau_gen_metric') is not None
)
tau_gen_metric_summary = ', '.join(sorted(tau_gen_metrics_used)) if tau_gen_metrics_used else 'N/A'

print('\n' + '='*65)
print('  TRACK 2 INTERPRETATION')
print('='*65)
print(f'  Dataset               : {CFG["dataset"]}  ({CFG["image_size"]}×{CFG["image_size"]})')
print(f'  Architecture          : DiT-{CFG["dit_variant"]}  (flow matching)')
vae_bl_best = vae_baseline_cache.get(best_c, float('nan'))
print(f'  Best d_latent (d*)    : {best_d}  (c={best_c})')
print(f'  VAE baseline FID (d*) : {vae_bl_best:.1f}')
print(f'  FLIPD intrinsic dim   : {d_int}  ({flipd_source_interp})')
print(f'  |d* − d_int| / d_int  : {abs(best_d - d_int) / (d_int + 1e-8) * 100:.1f}%')
print(f'  VAE baselines         : ' +
      '  '.join(f'c={c}:{vae_baseline_cache.get(c, float("nan")):.1f}' for c in cs))
print(f'  τ_gen metric(s) used  : {tau_gen_metric_summary}')
print('='*65)

---
## 11 · Save artefacts to GCS  *(skip on Kaggle)*

In [ ]:
GCS_BUCKET = os.environ.get('GCS_BUCKET', '')

if IN_GCP and GCS_BUCKET:
    import re
    from google.cloud import storage
    match = re.match(r'gs://([^/]+)/?(.*)', GCS_BUCKET)
    bucket_name, prefix = match.group(1), match.group(2)
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    for fname in os.listdir(OUT_DIR):
        if fname.endswith(('.png', '.csv', '.pt')):
            blob_path = f'{prefix}/{fname}' if prefix else fname
            bucket.blob(blob_path).upload_from_filename(
                os.path.join(OUT_DIR, fname))
            print(f'Uploaded: gs://{bucket_name}/{blob_path}')
elif IN_KAGGLE:
    print('Kaggle: artefacts in /kaggle/working/track2/ (visible in output tab).')
else:
    print(f'Local: artefacts in {OUT_DIR}\nSet GCS_BUCKET env var to upload.')